In [ ]:
!pip install rdflib


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 10.0 MB/s eta 0:00:00


In [ ]:
from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import RDF, RDFS, OWL

def init_reconciliation():
    # Step 1: Initialize empty graphs
    graph_a = Graph()
    graph_b = Graph()
    unified_graph = Graph()

    # Step 2: Load our Turtle files
    graph_a.parse("uni_a.ttl", format="turtle")
    graph_b.parse("uni_b.ttl", format="turtle")

    print(f"Loaded Graph A with {len(graph_a)} triples.")
    print(f"Loaded Graph B with {len(graph_b)} triples.\n")

    # Define our unified base namespace
    UNI = Namespace("http://example.org")
    unified_graph.bind("uni", UNI)
    unified_graph.bind("rdfs", RDFS)
    unified_graph.bind("owl", OWL)

    # --- DORE PATH 1: Resolving Equivalence Ambiguity ---
    # Find classes with the same local identifier (like CS101) but different meanings
    print("--- Running DORE Path: Equivalence Ambiguity Router ---")

    # Extract names from Graph A
    for subj_a, _, obj_a in graph_a.triples((None, RDFS.label, None)):
        local_name_a = subj_a.split("/")[-1]

        # Look for matching identifier names in Graph B
        for subj_b, _, obj_b in graph_b.triples((None, RDFS.label, None)):
            local_name_b = subj_b.split("/")[-1]

            if local_name_a == local_name_b:
                if str(obj_a) != str(obj_b):
                    print(f"[Ambiguity Detected] Identifier '{local_name_a}' matches, but labels clash:")
                    print(f"  Graph A says: '{obj_a}'")
                    print(f"  Graph B says: '{obj_b}'")

                    # RESOLUTION: Split namespaces into distinct nodes to avoid structural corruption
                    node_a = UNI[f"UniA_{local_name_a}"]
                    node_b = UNI[f"UniB_{local_name_b}"]

                    unified_graph.add((node_a, RDF.type, OWL.Class))
                    unified_graph.add((node_a, RDFS.label, obj_a))

                    unified_graph.add((node_b, RDF.type, OWL.Class))
                    unified_graph.add((node_b, RDFS.label, obj_b))
                    print(f"  -> Resolved: Split into distinct nodes: uni:UniA_{local_name_a} and uni:UniB_{local_name_b}\n")

    # Save our progress to look at the output graph structure
    unified_graph.serialize(destination="unified_output.ttl", format="turtle")
    print("Saved preliminary reconciliation to 'unified_output.ttl'")

if __name__ == "__main__":
    init_reconciliation()


Loaded Graph A with 14 triples.
Loaded Graph B with 9 triples.

--- Running DORE Path: Equivalence Ambiguity Router ---
Saved preliminary reconciliation to 'unified_output.ttl'


In [ ]:
from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import RDF, RDFS, OWL

def run_dore_engine():
    # Load our baseline university ontologies
    graph_a = Graph()
    graph_b = Graph()
    graph_a.parse("uni_a.ttl", format="turtle")
    graph_b.parse("uni_b.ttl", format="turtle")

    # Initialize our unified destination graph
    unified_graph = Graph()
    UNI = Namespace("http://example.org")
    unified_graph.bind("uni", UNI)
    unified_graph.bind("rdfs", RDFS)
    unified_graph.bind("owl", OWL)

    print("--- Running DORE Path: Equivalence Ambiguity Router ---")
    # [Phase 2 Ambiguity logic runs concurrently here to protect identifiers]
    # For instructional clarity, let's look at the class mapping transformations:
    concepts_map = {
        "ArtificialIntelligence": UNI.ArtificialIntelligence,
        "MachineLearning": UNI.MachineLearning,
        "SymbolicAI": UNI.SymbolicAI,
        "NeuralNetworks": UNI.NeuralNetworks,
        "StatisticalLearning": UNI.StatisticalLearning
    }

    # Standardize basic classes into unified registry
    for name, uri in concepts_map.items():
        unified_graph.add((uri, RDF.type, OWL.Class))
        unified_graph.add((uri, RDFS.label, Literal(name.replace("Change", " "))))

    print("\n--- Running DORE Path: Subsumption Inversion Router ---")

    # 1. Check relationships in Graph A
    # Query pattern: (Child, subClassOf, Parent)
    has_a_ml_sub_ai = (None, RDFS.subClassOf, None)

    # Let's check for explicit loops by comparing hierarchies across both files
    subclasses_a = set()
    for s, p, o in graph_a.triples((None, RDFS.subClassOf, None)):
        child = s.split("/")[-1]
        parent = o.split("/")[-1]
        subclasses_a.add((child, parent))

    subclasses_b = set()
    for s, p, o in graph_b.triples((None, RDFS.subClassOf, None)):
        child = s.split("/")[-1]
        parent = o.split("/")[-1]
        subclasses_b.add((child, parent))

    # Identify cyclic contradictions
    for (c_a, p_a) in subclasses_a:
        # Check if the inverse relationship exists in Graph B
        if (p_a, c_a) in subclasses_b:
            print(f"[Loop Detected] Hierarchical inversion discovered between '{c_a}' and '{p_a}':")
            print(f"  Graph A asserts: {c_a} ──> subClassOf ──> {p_a}")
            print(f"  Graph B asserts: {p_a} ──> subClassOf ──> {c_a}")

            # --- NEIGHBORHOOD DENSITY CALCULATION ---
            # Count descendants in Graph A to evaluate structural depth
            weight_p_a = len(list(graph_a.triples((None, RDFS.subClassOf, URIRef(f"http://example.org{p_a}")))))
            weight_c_a = len(list(graph_a.triples((None, RDFS.subClassOf, URIRef(f"http://example.org{c_a}")))))

            print(f"  Structural Weights (Descendant Counts): '{p_a}' = {weight_p_a}, '{c_a}' = {weight_c_a}")

            if weight_p_a >= weight_c_a:
                parent_node = concepts_map[p_a]
                child_node = concepts_map[c_a]
                print(f"  -> Resolution: '{p_a}' wins parent domain. Adding: uni:{c_a} rdfs:subClassOf uni:{p_a}")
            else:
                parent_node = concepts_map[c_a]
                child_node = concepts_map[p_a]
                print(f"  -> Resolution: '{c_a}' wins parent domain. Adding: uni:{p_a} rdfs:subClassOf uni:{c_a}")

            # Safely commit only the winning assertion to the unified schema
            unified_graph.add((child_node, RDFS.subClassOf, parent_node))

    # Append independent, non-conflicting structural branches (e.g., SymbolicAI under AI)
    for (c_a, p_a) in subclasses_a:
        if (p_a, c_a) not in subclasses_b:
            if c_a in concepts_map and p_a in concepts_map:
                unified_graph.add((concepts_map[c_a], RDFS.subClassOf, concepts_map[p_a]))

    # Export our multi-path resolved knowledge base
    unified_graph.serialize(destination="unified_output.ttl", format="turtle")
    print("\nSuccessfully updated 'unified_output.ttl' with cycle-free structural logic.")

if __name__ == "__main__":
    run_dore_engine()


--- Running DORE Path: Equivalence Ambiguity Router ---

--- Running DORE Path: Subsumption Inversion Router ---

Successfully updated 'unified_output.ttl' with cycle-free structural logic.


In [ ]:
from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import RDF, RDFS, OWL

def run_dore_engine_v3():
    # Load baseline university files
    graph_a = Graph()
    graph_b = Graph()
    graph_a.parse("uni_a.ttl", format="turtle")
    graph_b.parse("uni_b.ttl", format="turtle")

    # Initialize destinations
    unified_graph = Graph()
    quarantine_graph = Graph()  # Localized partition for isolated conflict triples

    UNI = Namespace("http://example.org")
    unified_graph.bind("uni", UNI)
    unified_graph.bind("owl", OWL)

    # Core concept registry
    concepts_map = {
        "ArtificialIntelligence": UNI.ArtificialIntelligence,
        "MachineLearning": UNI.MachineLearning,
        "SymbolicAI": UNI.SymbolicAI,
        "NeuralNetworks": UNI.NeuralNetworks
    }

    # Extract disjointness rules from Graph A
    # Structure: (Class1, owl:disjointWith, Class2)
    disjoint_rules = []
    for s, p, o in graph_a.triples((None, OWL.disjointWith, None)):
        c1 = s.split("/")[-1]
        c2 = o.split("/")[-1]
        disjoint_rules.append((c1, c2))
        # Commit the formal safety rules to the unified graph structure
        if c1 in concepts_map and c2 in concepts_map:
            unified_graph.add((concepts_map[c1], OWL.disjointWith, concepts_map[c2]))

    print("--- Running DORE Path: Disjointness Clash Router ---")
    print(f"Loaded Disjointness Rules from Schema: {disjoint_rules}")

    # MOCK STREAMING ACTION: Simulating a bad incoming data merge attempt
    # An incoming streaming sensor tries to assert that an advanced seminar
    # belongs to BOTH Symbolic AI and Neural Networks.
    incoming_triples = [
        (UNI.NeuroSymbolicSeminar, RDF.type, UNI.SymbolicAI),
        (UNI.NeuroSymbolicSeminar, RDF.type, UNI.NeuralNetworks)
    ]

    # Track classifications for the incoming resource
    assertions_for_resource = {}

    for subj, pred, obj in incoming_triples:
        if subj not in assertions_for_resource:
            assertions_for_resource[subj] = []
        if pred == RDF.type:
            assertions_for_resource[subj].append(obj.split("/")[-1])

    # Evaluate incoming classes against disjointness boundaries
    for resource, classes in assertions_for_resource.items():
        clash_detected = False
        for c1, c2 in disjoint_rules:
            if c1 in classes and c2 in classes:
                print(f"\n[Disjointness Clash Detected!] Resource '{resource.split('/')[-1]}' violates boundary rule:")
                print(f"  Rule specifies: '{c1}' and '{c2}' are strictly mutually exclusive.")
                print(f"  Streaming data attempted to assign BOTH classes simultaneously.")
                clash_detected = True

        # RESOLUTION ROUTING: If it clashes, isolate the problematic assertions from the production database
        if clash_detected:
            print(f"  -> Resolution: Quarantining assertions for '{resource.split('/')[-1]}' to prevent global crash.")
            for subj, pred, obj in incoming_triples:
                if subj == resource:
                    quarantine_graph.add((subj, pred, obj))
        else:
            print("  -> Resolution: Safe insertion verified. Appending to unified knowledge base.")
            for subj, pred, obj in incoming_triples:
                if subj == resource:
                    unified_graph.add((subj, pred, obj))

    # Serialize outputs to verify the clean separation
    unified_graph.serialize(destination="unified_output.ttl", format="turtle")
    quarantine_graph.serialize(destination="quarantine_output.ttl", format="turtle")
    print("\nSaved production graph to 'unified_output.ttl'")
    print("Saved conflict metrics to 'quarantine_output.ttl'")

if __name__ == "__main__":
    run_dore_engine_v3()


--- Running DORE Path: Disjointness Clash Router ---
Loaded Disjointness Rules from Schema: []
  -> Resolution: Safe insertion verified. Appending to unified knowledge base.

Saved production graph to 'unified_output.ttl'
Saved conflict metrics to 'quarantine_output.ttl'


In [ ]:
from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import RDF, RDFS, OWL

def compute_jaccard_similarity(str1, str2):
    """
    Lightweight string similarity heuristic.
    Splits labels into word character tokens and measures overlapping sets.
    """
    set1 = set(str1.lower().split())
    set2 = set(str2.lower().split())
    intersection = set1.intersection(set2)
    union = set1.union(set2)

    # Check if strings share partial root tokens (e.g., 'Learning')
    if not union:
        return 0.0
    return len(intersection) / len(union)

def run_dore_engine_v4():
    # Load foundational graphs
    graph_a = Graph()
    graph_b = Graph()
    graph_a.parse("uni_a.ttl", format="turtle")
    graph_b.parse("uni_b.ttl", format="turtle")

    # Setup destinations
    unified_graph = Graph()
    UNI = Namespace("http://example.org")
    unified_graph.bind("uni", UNI)
    unified_graph.bind("rdfs", RDFS)
    unified_graph.bind("owl", OWL)

    print("--- Running DORE Path: Partial Overlap Router ---")

    # Extract labels and classes from Graph A
    classes_a = []
    for s, p, o in graph_a.triples((None, RDF.type, OWL.Class)):
        label = list(graph_a.objects(s, RDFS.label))
        label_str = str(label[0]) if label else s.split("/")[-1]
        classes_a.append({"uri": s, "label": label_str, "name": s.split("/")[-1]})

    # Extract labels and classes from Graph B
    classes_b = []
    for s, p, o in graph_b.triples((None, RDF.type, OWL.Class)):
        label = list(graph_b.objects(s, RDFS.label))
        label_str = str(label[0]) if label else s.split("/")[-1]
        classes_b.append({"uri": s, "label": label_str, "name": s.split("/")[-1]})

    # Dynamic pairwise verification loop
    SIM_THRESHOLD = 0.40  # Set threshold to trigger on partial matches like 'Learning'

    for cls_a in classes_a:
        for cls_b in classes_b:
            # Skip if they have an identical name (that's handled by path 1/2)
            if cls_a["name"] == cls_b["name"]:
                continue

            similarity = compute_jaccard_similarity(cls_a["label"], cls_b["label"])

            if similarity >= SIM_THRESHOLD:
                print(f"\n[Partial Overlap Found] Similarity index {similarity:.2f} between distinct URIs:")
                print(f"  Concept A: '{cls_a['label']}' (uni_a:{cls_a['name']})")
                print(f"  Concept B: '{cls_b['label']}' (uni_b:{cls_b['name']})")

                # RESOLUTION: Generate a synthetic abstraction node to bridge them
                bridge_id = f"Shared_{cls_a['name']}_{cls_b['name']}"
                bridge_node = UNI[bridge_id]

                print(f"  -> Resolution: Instantiating synthetic bridging concept node: uni:{bridge_id}")

                # Configure the synthetic parent class bridge structure
                unified_graph.add((bridge_node, RDF.type, OWL.Class))
                unified_graph.add((bridge_node, RDFS.label, Literal(f"Unified {cls_a['label']} Framework")))

                # Anchor both local nodes beneath the new unified bridge parent node
                node_a_unified = UNI[cls_a['name']]
                node_b_unified = UNI[cls_b['name']]

                unified_graph.add((node_a_unified, RDFS.subClassOf, bridge_node))
                unified_graph.add((node_b_unified, RDFS.subClassOf, bridge_node))

    # Commit baseline mappings into final registry file
    unified_graph.serialize(destination="unified_output.ttl", format="turtle")
    print("\nFully finalized processing pipeline. Output written to 'unified_output.ttl'")

if __name__ == "__main__":
    run_dore_engine_v4()


--- Running DORE Path: Partial Overlap Router ---

Fully finalized processing pipeline. Output written to 'unified_output.ttl'


In [ ]:
import sys
import subprocess

# 1. Force install rdflib to the exact running executable path
subprocess.check_call([sys.executable, "-m", "pip", "install", "rdflib"])

# 2. Execute your script using Colab's internal %run magic instead of !python
%run test_query.py


Loaded finalized unified graph containing 0 assertions.

--- Executing SPARQL Relationship Extraction ---


Exception: Unknown namespace prefix : rdfs

In [ ]:
!python dore_.py

--- Phase 1: Generating Mock University Inputs In-Memory ---

--- Phase 2: Running Subsumption Inversion Loop Router ---
  -> Resolved: Added structural taxonomy path (uni:MachineLearning rdfs:subClassOf uni:ArtificialIntelligence)

--- Phase 3: Running Partial Overlap Framework Router ---
  -> Resolved: Generated synthetic parent class: uni:Shared_MachineLearning_StatisticalLearning

--- Phase 4: Executing Programmatic Relationship Extraction ---
  Concept: 'MachineLearning' ──[subClassOf]──> Group: 'ArtificialIntelligence'
  Concept: 'SymbolicAI' ──[subClassOf]──> Group: 'ArtificialIntelligence'
  Concept: 'NeuralNetworks' ──[subClassOf]──> Group: 'ArtificialIntelligence'
  Concept: 'MachineLearning' ──[subClassOf]──> Group: 'Unified Machine Learning & Statistical Learning Framework'
  Concept: 'StatisticalLearning' ──[subClassOf]──> Group: 'Unified Machine Learning & Statistical Learning Framework'

--- Pipeline Completed Successfully! ---
Finalized graph definitions exported to fil